In [ ]:
"""Example problem: trying to run x^2 + 1 (with zero noise)
"""

Following: simopt/models/example.py 

In [14]:
from __future__ import annotations

from typing import Annotated, ClassVar

import numpy as np
from pydantic import BaseModel, Field

from mrg32k3a.mrg32k3a import MRG32k3a
from simopt import dsl
from simopt.base import (
    ConstraintType,
    Model,
    Problem,
    VariableType,
)
from simopt.input_models import Normal
# from simopt.simulations.example import (
#     Example2ModelConfig,
#     ExampleModelConfig,
#     replicate,
#     replicate_discrete,
# )
from simopt.utils import override

# Like in simulation 
class QuadNoiseConfig(BaseModel):
    x: Annotated[
        tuple[float, ...],
        Field(default=(5.0,), description="decision variable"),
    ]
    noise_std: Annotated[
        float,
        Field(default=1e-8, ge=0.0, description="tiny noise standard deviation"),
    ]

class QuadNoiseModel(Model[QuadNoiseConfig]):
    """x^2 + 1 evaluated with tiny stochastic noise."""

    class_name_abbr: ClassVar[str] = "QUADNOISE"
    class_name: ClassVar[str] = "x^2 + 1 with tiny noise"
    config_class: ClassVar[type[QuadNoiseConfig]] = QuadNoiseConfig
    n_rngs: ClassVar[int] = 1
    n_responses: ClassVar[int] = 1
    
    def __init__(self, fixed_factors: dict | None = None) -> None:
        super().__init__(fixed_factors)
        self.noise_model = Normal()
        
    def replicate(self, factors: QuadNoiseConfig, rngs: list[MRG32k3a]) -> tuple[dict, dict]:
        x = np.asarray(factors.x, dtype=float)
        f = float(np.sum(x**2) + 1.0)

        # Tiny noise: set noise_std = 0.0 for exact deterministic behavior.
        noise = self.noise_model.random(rngs[0], mu=0.0, sigma=factors.noise_std)
        noisy_value = f + noise

        responses = {"est_f(x)": noisy_value}
        gradients = {"est_f(x)": {"x": 2.0 * x}}
        return responses, gradients
    
    #Similar to example.py in models.   
class QuadNoiseProblemConfig(BaseModel):
    initial_solution: Annotated[
        tuple[float, ...],
        Field(default=(5.0,), description="initial point"),
    ]
    budget: Annotated[
        int,
        Field(default=1000, gt=0, description="max simulation budget"),
    ]
    
class QuadNoiseProblem(Problem):
    """Minimize x^2 + 1 under noisy stochastic replications."""

    class_name_abbr: ClassVar[str] = "QUADNOISE"
    class_name: ClassVar[str] = "Minimize x^2 + 1 with zero noise"
    config_class: ClassVar[type[BaseModel]] = QuadNoiseProblemConfig
    model_class: ClassVar[type[Model]] = QuadNoiseModel
    n_objectives: ClassVar[int] = 1
    n_stochastic_constraints: ClassVar[int] = 0
    minmax: ClassVar[tuple[int, ...]] = (-1,)
    constraint_type: ClassVar[ConstraintType] = ConstraintType.UNCONSTRAINED
    variable_type: ClassVar[VariableType] = VariableType.CONTINUOUS
    gradient_available: ClassVar[bool] = True
    model_default_factors: ClassVar[dict] = {}
    model_decision_factors: ClassVar[set[str]] = {"x"}
    
    @property
    def optimal_value(self) -> float | None:
        return 1.0
    
    @property
    def optimal_solution(self) -> tuple | None:
        return (0.0,)
    
    @override
    def build(self) -> dsl.Model:
        problem = dsl.Model()
        x0 = self.factors["initial_solution"]
        x = problem.add_continuous_vector(
            lb=-10.0, ub=10.0, shape=(len(x0),), initial=x0,
        )
        simulation = self.add_simulation(problem, {"x": x})
        problem.minimize(dsl.mean(simulation.metric("est_f(x)")))
        return problem
    
    def vector_to_factor_dict(self, vector: tuple) -> dict:
        return {"x": vector[:]}
    
    def get_random_solution(self, rand_sol_rng: MRG32k3a) -> tuple:
        # unchanged from the example
        return tuple(
            rand_sol_rng.mvnormalvariate(
                mean_vec=[0.0] * self.dim,
                cov=np.eye(self.dim).tolist(),
                factorized=False,
            )
        )

In [19]:
from simopt.experiment_base import ProblemSolver
from simopt.solvers.astrodf import ASTRODF

exp = ProblemSolver(
    solver=ASTRODF(),
    problem=QuadNoiseProblem(
        fixed_factors={"initial_solution": (1.0,), "budget": 1000},
    ),
    solver_fixed_factors={"use_gradients": False},
)
exp.run(n_macroreps=2)
exp.post_replicate(n_postreps=50)
print("done")

done


In [20]:
import numpy as np
from simopt.solvers.astrodf import ASTRODF

class DebugASTRODF(ASTRODF):
    def iterate(self, ctx):
        self.iteration_count += 1

        pilot_run = int(np.ceil(
            max(
                self.lambda_min * np.log10(10 + self.iteration_count) ** 1.1,
                min(0.5 * self.problem.dim, ctx.budget.total),
            )
            - 1
        ))

        if self.iteration_count == 1:
            self.incumbent_solution = ctx.evaluate(self.incumbent_x, 0)
            self.visited_pts_list.append(self.incumbent_solution)
            self.perform_adaptive_sampling(
                ctx,
                self.incumbent_solution,
                pilot_run,
                self.delta_k,
                compute_kappa=True,
            )
            ctx.log(self.incumbent_solution)
        elif self.factors["crn_across_solns"]:
            self.perform_adaptive_sampling(ctx, self.incumbent_solution, pilot_run, self.delta_k)

        if self.enable_gradient:
            fval = np.ones(2 * self.problem.dim + 1) * self.incumbent_solution.objectives_mean
            grad = self.incumbent_solution.objectives_gradients_mean[0]
            hessian = self.h_k
            q = np.array([])
            y_var = [[]]
            interpolation_solns = []
        else:
            (
                fval,
                y_var,
                q,
                grad,
                hessian,
                interpolation_solns,
            ) = self.construct_model(ctx)

        if self.easy_solve:
            dot_a = np.dot(grad, hessian) if self.enable_gradient else grad * hessian
            check_positive_definite = np.dot(dot_a, grad)

            if check_positive_definite <= 0:
                tau = 1.0
            else:
                norm_ratio = np.linalg.norm(grad) ** 3 / (self.delta_k * check_positive_definite)
                tau = min(1.0, float(norm_ratio))

            grad = np.reshape(grad, (1, self.problem.dim))[0]
            grad_norm = np.linalg.norm(grad)
            if grad_norm == 0:
                candidate_x = self.incumbent_x
            else:
                product = tau * self.delta_k * grad
                adjustment = product / float(grad_norm)
                candidate_x = self.incumbent_x - adjustment
        else:
            def subproblem(s):
                s_grad_dot = np.dot(s, grad)
                s_hessian_dot = np.dot(np.multiply(s, hessian), s)
                return float((fval[0] + s_grad_dot + s_hessian_dot)[0])

            def con_f(s):
                return float(np.linalg.norm(s))

            nlc = __import__("scipy.optimize").optimize.NonlinearConstraint(con_f, 0, self.delta_k)
            solve_subproblem = __import__("scipy.optimize").optimize.minimize(
                subproblem,
                np.zeros(self.problem.dim),
                method="trust-constr",
                constraints=nlc,
            )
            candidate_x = self.incumbent_x + solve_subproblem.x

        candidate_x = tuple(
            float(x) for x in candidate_x
        )

        candidate_solution = ctx.evaluate(candidate_x, 0)
        self.visited_pts_list.append(candidate_solution)

        if self.factors["crn_across_solns"]:
            num_sims = self.incumbent_solution.n_reps
            candidate_solution = ctx.evaluate(candidate_solution, num_sims)
        else:
            self.perform_adaptive_sampling(ctx, candidate_solution, pilot_run, self.delta_k)

        fval_tilde = candidate_solution.objectives_mean

        if not self.enable_gradient:
            min_fval = min(fval)
            sufficient_reduction = (fval[0] - min_fval) >= self.factors["ps_sufficient_reduction"] * self.delta_k**2
            condition_met = min_fval < fval_tilde and sufficient_reduction

            high_variance = False
            if not condition_met:
                if candidate_solution.objectives_mean[0] == 0:
                    pass
                else:
                    high_variance = (
                        candidate_solution.objectives_var[0]
                        / (candidate_solution.n_reps * candidate_solution.objectives_mean[0] ** 2)
                    ) > 0.75

            if condition_met or high_variance:
                fval_tilde = min_fval
                min_idx = np.argmin(fval)
                candidate_x = y_var[min_idx][0]
                candidate_solution = interpolation_solns[min_idx]

        candidate_x_arr = np.array(candidate_x)
        incumbent_x_arr = np.array(self.incumbent_x)
        s = np.subtract(candidate_x_arr, incumbent_x_arr)

        if self.enable_gradient:
            model_reduction = -np.dot(s, grad) - 0.5 * np.dot(np.dot(s, hessian), s)
        else:
            model_reduction = self.evaluate_model(np.zeros(self.problem.dim), q) - self.evaluate_model(s, q)

        rho = 0 if model_reduction <= 0 else (fval[0] - fval_tilde) / model_reduction

        print(f"\nITER={self.iteration_count}")
        print(f"delta_k={self.delta_k}")
        print(f"incumbent_x={self.incumbent_x}")
        print(f"grad={np.asarray(grad).reshape(-1)}")
        print(f"hessian={np.asarray(hessian).reshape(-1)}")
        print(f"candidate_x={tuple(np.asarray(candidate_x).reshape(-1))}")
        print(f"rho={rho}")

        successful = rho >= self.eta_1
        if successful:
            self.incumbent_x = candidate_x
            self.incumbent_solution = candidate_solution
            ctx.log(candidate_solution)
            self.delta_k = min(self.delta_k, self.delta_max)
            if rho >= self.eta_2:
                self.delta_k = min(self.gamma_1 * self.delta_k, self.delta_max)
            if self.enable_gradient:
                self.update_hessian(candidate_solution, grad, s)
        else:
            self.delta_k = min(self.gamma_2 * self.delta_k, self.delta_max)

In [21]:
exp = ProblemSolver(
    solver=DebugASTRODF(fixed_factors={"use_gradients": True}),
    problem=QuadNoiseProblem(
        fixed_factors={"initial_solution": (1.0,), "budget": 1000},
    ),
)

exp.run(n_macroreps=1)
exp.post_replicate(n_postreps=20)


ITER=1
delta_k=10.0
incumbent_x=(1.0,)
grad=[2.]
hessian=[1.]
candidate_x=(np.float64(-1.0),)
rho=[0.]

ITER=2
delta_k=5.0
incumbent_x=(1.0,)
grad=[2.]
hessian=[1.]
candidate_x=(np.float64(-1.0),)
rho=[0.]

ITER=3
delta_k=2.5
incumbent_x=(1.0,)
grad=[2.]
hessian=[1.]
candidate_x=(np.float64(-1.0),)
rho=[0.]

ITER=4
delta_k=1.25
incumbent_x=(1.0,)
grad=[2.]
hessian=[1.]
candidate_x=(np.float64(-0.25),)
rho=[0.54545455]

ITER=5
delta_k=1.25
incumbent_x=(-0.25,)
grad=[-0.5]
hessian=[2.]
candidate_x=(np.float64(0.0),)
rho=[1.]

ITER=6
delta_k=3.125
incumbent_x=(0.0,)
grad=[0.]
hessian=[2.]
candidate_x=(np.float64(0.0),)
rho=0

ITER=7
delta_k=1.5625
incumbent_x=(0.0,)
grad=[0.]
hessian=[2.]
candidate_x=(np.float64(0.0),)
rho=0

ITER=8
delta_k=0.78125
incumbent_x=(0.0,)
grad=[0.]
hessian=[2.]
candidate_x=(np.float64(0.0),)
rho=0

ITER=9
delta_k=0.390625
incumbent_x=(0.0,)
grad=[0.]
hessian=[2.]
candidate_x=(np.float64(0.0),)
rho=0

ITER=10
delta_k=0.1953125
incumbent_x=(0.0,)
grad=[0.]
hess